# LightGBM LambdaMART

Issue: #23

## Question

Does a gradient-boosted ranker, trained on the features from #22, beat co-visitation when it reorders the shared candidate pool?

## Data used

Inputs: `data/features/train_asof11.parquet` (40,000 training queries, features as of day 11) and `data/features/eval_w0.parquet` (the 56,544 evaluation queries, as of day 14) from #22; the shared rules and metrics from #17; results of #17 and #18 for comparison.

Outputs: `data/models/lgbm_lambdarank.txt`, `lgbm_params.json`, `data/results/lgbm.parquet` and `lgbm_binary.parquet`.

**Setup.** Each query has a pool of 200 candidates. Labels: 0 not found later, 1 click, 2 cart, 3 order. The model scores every candidate and the pool is re-ordered by score; the top 100 are the result list. Only queries with at least one positive candidate are used for **training** (a query with no positive gives no ranking signal: every pair has the same label, so its gradients are zero). **Evaluation** uses all 56,544 queries, so queries whose pool has no positive count as zero, as for every other method.

## How lambdarank works, in short

A pairwise ranker looks at pairs of candidates (i, j) in one query where i should be above j. RankNet gives each pair a gradient that pushes the score of i up and j down, larger when the model has them in the wrong order. But NDCG only cares about the top of the list, and it is not differentiable. **LambdaRank** fixes this by multiplying each pair's gradient by `|change in NDCG|` if i and j swapped places. Swapping two items near the top, or with a big gain difference (an order versus a click), changes NDCG a lot, so those pairs get large 'lambda' gradients; swapping two items deep in the list changes almost nothing, so they get small ones. **LambdaMART** is this gradient used inside gradient-boosted trees: each new tree fits the lambdas. So the model optimises what NDCG rewards without ever differentiating NDCG. Here `label_gain = [0, 1, 2, 3]` makes LightGBM's gains match the linear gains of our own NDCG.

In [ ]:
import json
import resource
import sys
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import polars as pl

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from bootstrap import paired_bootstrap_diff
from evalset import TOP, evaluate, prepare_eval, summarize
from features import FEATURE_COLUMNS

DATA = Path("data") if Path("data").exists() else Path("../data")
FEAT = DATA / "features"
MODELS = DATA / "models"
RESULTS = DATA / "results"
MODELS.mkdir(parents=True, exist_ok=True)
SEED = 0
POOL = 200
print(len(FEATURE_COLUMNS), "features:", FEATURE_COLUMNS)

## Training data

Sessions are split 90/10 into train and validation. Early stopping uses the **validation** part only, never the evaluation queries.

In [ ]:
train_lf = pl.scan_parquet(FEAT / "train_asof11.parquet")
with_pos = train_lf.group_by("session").agg(pl.col("label").max().alias("best")).filter(pl.col("best") > 0).select("session").collect()
all_sessions = train_lf.select("session").unique().collect().height
print(f"training queries: {all_sessions:,}; with at least one positive candidate: {with_pos.height:,} ({with_pos.height / all_sessions:.1%})")

df = train_lf.join(with_pos.lazy(), on="session").collect().sort("session")
sessions = np.sort(with_pos["session"].to_numpy())
rng = np.random.default_rng(SEED)
valid_sessions = set(rng.choice(sessions, size=len(sessions) // 10, replace=False).tolist())
is_valid = df["session"].is_in(list(valid_sessions)).to_numpy()
tr, va = df.filter(~pl.Series(is_valid)), df.filter(pl.Series(is_valid))
assert not (set(tr["session"].to_list()) & set(va["session"].to_list())), "train and validation sessions overlap"
eval_sessions = set(pl.scan_parquet(FEAT / "eval_w0.parquet").select("session").unique().collect()["session"].to_list())
assert not (set(df["session"].to_list()) & eval_sessions), "a training session is also an evaluation session"
for name, part in (("train", tr), ("validation", va)):
    counts = part.group_by("session", maintain_order=True).len()["len"]
    assert (counts == POOL).all()
    print(f"{name}: {part['session'].n_unique():,} queries, {part.height:,} rows")


def xy(part, label):
    return part.select(FEATURE_COLUMNS).to_numpy().astype(np.float32), part[label].to_numpy()


X_tr, y_tr = xy(tr, "label")
X_va, y_va = xy(va, "label")
groups_tr = [POOL] * (tr.height // POOL)
groups_va = [POOL] * (va.height // POOL)
# rows of one session are contiguous (sorted by session) and each group has 200 rows
assert (tr["session"].to_numpy()[::POOL] == tr["session"].to_numpy()[POOL - 1 :: POOL]).all()

## Train with the lambdarank objective

In [ ]:
params = {
    "objective": "lambdarank", "metric": "ndcg", "ndcg_eval_at": [10], "label_gain": [0, 1, 2, 3],
    "learning_rate": 0.05, "num_leaves": 63, "min_data_in_leaf": 100, "feature_fraction": 0.8,
    "bagging_fraction": 0.8, "bagging_freq": 1, "lambdarank_truncation_level": 20, "max_bin": 127,
    "seed": SEED, "deterministic": True, "force_col_wise": True, "verbose": -1,
}
d_tr = lgb.Dataset(X_tr, y_tr, group=groups_tr, feature_name=FEATURE_COLUMNS, categorical_feature=["category"], free_raw_data=False)
d_va = lgb.Dataset(X_va, y_va, group=groups_va, reference=d_tr, free_raw_data=False)
t = time.time()
rank_model = lgb.train(params, d_tr, num_boost_round=600, valid_sets=[d_va], valid_names=["validation"],
                       callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(50)])
print(f"lambdarank: best iteration {rank_model.best_iteration}, validation NDCG@10 {rank_model.best_score['validation']['ndcg@10']:.4f}, trained in {time.time() - t:.0f} s")
rank_model.save_model(str(MODELS / "lgbm_lambdarank.txt"), num_iteration=rank_model.best_iteration)
(MODELS / "lgbm_params.json").write_text(json.dumps({"lambdarank": params, "best_iteration": rank_model.best_iteration, "features": FEATURE_COLUMNS}, indent=2))

## Ablation: binary logloss on click versus no click

Same features, same split, a plain classifier on `y_click`. It does not know about the list, only about single items.

In [ ]:
bin_params = {
    "objective": "binary", "metric": "binary_logloss", "learning_rate": 0.05, "num_leaves": 63, "min_data_in_leaf": 100,
    "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "max_bin": 127,
    "seed": SEED, "deterministic": True, "force_col_wise": True, "verbose": -1,
}
_, yb_tr = xy(tr, "y_click")
_, yb_va = xy(va, "y_click")
db_tr = lgb.Dataset(X_tr, yb_tr, feature_name=FEATURE_COLUMNS, categorical_feature=["category"], free_raw_data=False)
db_va = lgb.Dataset(X_va, yb_va, reference=db_tr, free_raw_data=False)
t = time.time()
bin_model = lgb.train(bin_params, db_tr, num_boost_round=600, valid_sets=[db_va], valid_names=["validation"],
                      callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(100)])
print(f"binary: best iteration {bin_model.best_iteration}, trained in {time.time() - t:.0f} s")
bin_model.save_model(str(MODELS / "lgbm_binary.txt"), num_iteration=bin_model.best_iteration)
del d_tr, d_va, db_tr, db_va, X_tr, X_va, df, tr, va

## Score the evaluation queries and rank the pool

In [ ]:
ev = pl.read_parquet(FEAT / "eval_w0.parquet")
queries = pl.read_parquet(DATA / "eval" / "ranking_queries.parquet")
prepared, prep_stats = prepare_eval(queries)
n_q = len(prepared)
assert ev.height == n_q * POOL
assert ev["session"].to_numpy()[::POOL].tolist() == [q["session"] for q in prepared], "row order differs from the query order"
X_ev = ev.select(FEATURE_COLUMNS).to_numpy().astype(np.float32)
aids = ev["aid"].to_numpy().reshape(n_q, POOL)
row_of = {q["session"]: i for i, q in enumerate(prepared)}


def make_ranker(model):
    scores = model.predict(X_ev, num_iteration=model.best_iteration).reshape(n_q, POOL)
    order = np.argsort(-scores, axis=1, kind="stable")[:, :TOP]
    ranked = np.take_along_axis(aids, order, axis=1)
    return lambda q: [int(a) for a in ranked[row_of[q["session"]]]]


t = time.time()
res_rank = evaluate(prepared, make_ranker(rank_model))
res_bin = evaluate(prepared, make_ranker(bin_model))
print(f"scored and evaluated in {time.time() - t:.0f} s")
print(prep_stats)

In [ ]:
def show(name, r):
    s = summarize(r)
    print(name)
    for k, v in s.items():
        print(f"  {k:>22}: {v[0]:.4f}  [{v[1]:.4f}, {v[2]:.4f}]" if isinstance(v, tuple) else f"  {k:>22}: {v:.4f}  (OTTO official)")


show("LightGBM lambdarank", res_rank)
show("LightGBM binary logloss (ablation)", res_bin)
covis = pl.read_parquet(RESULTS / "covis.parquet")
pop = pl.read_parquet(RESULTS / "popularity.parquet")
for name, other in (("co-visitation (#18)", covis), ("popularity (#17)", pop)):
    j = res_rank.join(other.select("session", pl.col("ndcg10").alias("o")), on="session")
    assert j.height == res_rank.height
    m, lo, hi = paired_bootstrap_diff(j["ndcg10"].to_numpy(), j["o"].to_numpy())
    print(f"NDCG@10 lambdarank minus {name}: {m:+.4f}  [{lo:+.4f}, {hi:+.4f}]")
j = res_rank.join(res_bin.select("session", pl.col("ndcg10").alias("o")), on="session")
m, lo, hi = paired_bootstrap_diff(j["ndcg10"].to_numpy(), j["o"].to_numpy())
print(f"NDCG@10 lambdarank minus binary logloss: {m:+.4f}  [{lo:+.4f}, {hi:+.4f}]")

## Feature importance (gain), top 15

In [ ]:
imp = pl.DataFrame({"feature": rank_model.feature_name(), "gain": rank_model.feature_importance("gain")}).sort("gain", descending=True)
imp = imp.with_columns((pl.col("gain") / pl.col("gain").sum()).alias("share"))
print(imp.head(15))

In [ ]:
covis_like = ["rank_covis", "covis_max", "covis_mean", "covis_wsum", "rrf"]
share = imp.filter(pl.col("feature").is_in(covis_like))["share"].sum()
print(f"share of total gain from the co-visitation based features {covis_like}: {share:.1%}")

In [ ]:
res_rank.write_parquet(RESULTS / "lgbm.parquet")
res_bin.write_parquet(RESULTS / "lgbm_binary.parquet")
assert (res_rank["n_returned"] == TOP).all()
print("saved lgbm.parquet and lgbm_binary.parquet;", res_rank.height, "sessions")
print("peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))

## Result

Numbers are in the printouts above (mean and 95% bootstrap interval over the 56,544 evaluation queries).

- **Lambdarank beats co-visitation, but only just.** NDCG@10 0.2153 vs 0.2144: a paired difference of +0.0009 with an interval that excludes zero ([+0.0003, +0.0015]). Statistically real, practically tiny. It beats popularity by a wide margin (+0.173).
- **The binary-logloss ablation is as good or slightly better** (0.2175; lambdarank minus binary = -0.0022 [-0.0030, -0.0015]). Here the list-aware objective did not help. Only a third of queries give training signal, and the pool is short (200), so a per-item click model does about as well.
- **Why the gain is small.** About 56% of the total gain goes to `rank_covis` alone, and the five co-visitation based features together (`rank_covis`, `covis_max`, `covis_mean`, `covis_wsum`, `rrf`) carry 81% of the total gain. The pool is built from co-visitation, so the model largely learns to re-use the order it was handed. The other features (popularity, rates, recency, session features) add little on top. The pool also caps the result: the best possible NDCG@10 is 0.62 and a third of queries have no relevant item in the pool at all.
- **Early stopping at 22 rounds** (validation NDCG@10 0.320 on queries that have a positive) suggests the model overfits quickly or that training and evaluation features differ, as flagged in #22 (`hours_since_last_seen` and covis magnitudes shift). The validation number is not comparable with the evaluation NDCG: validation uses only queries with a positive candidate, evaluation uses all queries.
- Not done here: hyperparameter tuning, more training queries, or dropping the constant columns (`category_conf`). A stronger ranker probably needs candidates that co-visitation does not already order well (the Two-Tower in #24 to #26) and features beyond the covis signal.

## What I learned

To be written by the owner of the project.